# 02.3 CNN Classification

This notebook connects image shapes, transforms, a custom image dataset, a CNN model, and a training loop into one runnable classification workflow. The sklearn digits dataset is small, but it is enough to practice the same structure used in larger image projects.

The key idea is that the CNN should receive batched image tensors shaped `(N, C, H, W)` and return class logits shaped `(N, num_classes)`.

## Learning Goals

After this notebook, you should be able to:

1. Use a CNN for image classification.
2. Organize image data into `(N, C, H, W)`.
3. Write a complete CNN training pipeline.
4. val loss and accuracy.
5. Run a final evaluation on the test set.
6. Run inference on one or a few images.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


## Load the Data

`digits.images.shape == (N, 8, 8)` means this is a grayscale image dataset.


In [ ]:
digits = load_digits()
images = digits.images
labels = digits.target

print("images.shape =", images.shape)
print("labels.shape =", labels.shape)
print("num classes =", len(digits.target_names))

In [ ]:
plt.figure(figsize=(8, 3))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(images[i], cmap="gray")
    plt.title(f"label / label: {labels[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## Split Train, Validation, and Test

We again use a two-step split:

1. first split off the test set
2. then split validation from the training portion

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    images,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full,
)

print("train:", X_train.shape)
print("val:", X_val.shape)
print("test:", X_test.shape)

## Define Transforms

The pixel range is originally around `0~16`. Here we first scale it to `0~1`, then normalize with training-set statistics. 
The original pixel range is roughly `0~16`, so we first scale it to `0~1`, then normalize it using training-set statistics.

In [ ]:
train_mean = float(X_train.mean() / 16.0)
train_std = float(X_train.std() / 16.0)

print("train_mean =", train_mean)
print("train_std =", train_std)

In [ ]:
transform = transforms.Compose([
    transforms.Lambda(lambda x: x.float() / 16.0),
    transforms.Normalize(mean=[train_mean], std=[train_std]),
])

## Custom Image Dataset

This dataset is responsible for two things:

1. turn one `(H, W)` image into `(1, H, W)`
2. apply the transform

In [ ]:
class DigitsDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = torch.tensor(self.images[index], dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(self.labels[index], dtype=torch.long)

        if self.transform is not None:
            image = self.transform(image)

        return image, label


train_ds = DigitsDataset(X_train, y_train, transform=transform)
val_ds = DigitsDataset(X_val, y_val, transform=transform)
test_ds = DigitsDataset(X_test, y_test, transform=transform)

img0, label0 = train_ds[0]
print("img0.shape =", img0.shape)
print("label0 =", label0)
print("img0.min(), img0.max() =", img0.min().item(), img0.max().item())

In [ ]:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

xb, yb = next(iter(train_loader))
print("xb.shape =", xb.shape)
print("yb.shape =", yb.shape)

## Define the CNN Model

Because the input images are only `8x8`, the model does not need to be large.


In [ ]:
class DigitsCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 2 * 2, 32),
            nn.ReLU(),
            nn.Linear(32, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = DigitsCNN()
print(model)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print(loss_fn)
print(optimizer)

## Define Training and Evaluation Functions

This follows the same idea as the earlier training-loop notebook, except now the model is a CNN.


In [ ]:
def batch_accuracy(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_acc += batch_accuracy(logits, yb)
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches

## Start Training

Because `digits` is small, 12 epochs are enough to see clear results.


In [ ]:
history = []

for epoch in range(1, 13):
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }
    )

    if epoch == 1 or epoch % 4 == 0:
        print(
            f"epoch={epoch:02d} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

In [ ]:
history_df = pd.DataFrame(history)
print(history_df)

## Evaluate on the Test Set

As before, the test set is used only for the final evaluation.


In [ ]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb)
        preds = logits.argmax(dim=1)
        all_preds.append(preds)
        all_targets.append(yb)

all_preds = torch.cat(all_preds)
all_targets = torch.cat(all_targets)

test_acc = accuracy_score(all_targets.numpy(), all_preds.numpy())
cm = confusion_matrix(all_targets.numpy(), all_preds.numpy())

print("test accuracy =", test_acc)
print("confusion matrix =\n", cm)

## Run Inference on a Few Test Images

Here we simply inspect predictions for a few test images.


In [ ]:
sample_images = []
sample_labels = []

for i in range(6):
    img, label = test_ds[i]
    sample_images.append(img)
    sample_labels.append(label.item())

sample_batch = torch.stack(sample_images)

with torch.no_grad():
    logits = model(sample_batch)
    preds = logits.argmax(dim=1).tolist()

plt.figure(figsize=(9, 4))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(sample_images[i].squeeze(0), cmap="gray")
    plt.title(f"true={sample_labels[i]}, pred={preds[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Exercise 1
#
# Experiment with batch size.
#
# Change batch_size from 32 to 64, retrain, and compare the results. When you
# compare, look at both runtime and validation/test accuracy. Larger batches
# usually reduce the number of steps per epoch, but they can also change
# optimization behavior.

Exercise 1 Reference Note

Larger batches usually reduce the number of steps per epoch and can run faster per epoch, but they also change optimization behavior. Compare runtime, train accuracy, and validation accuracy rather than only final accuracy.

In [ ]:
# Exercise 2
#
# Experiment with CNN capacity.
#
# Change the first convolution's out_channels from 8 to 16, then retrain. Compare
# training accuracy and validation accuracy. If training improves but validation
# does not, the added capacity may not be helping generalization.

Exercise 2 Reference Note

Increasing the first convolution's channels gives the model more capacity. If validation improves, the original model may have been too small; if only training improves, the added capacity may be overfitting.

In [ ]:
# Exercise 3
#
# Answer in one or two full sentences:
# Why are image classification inputs often shaped as (N, C, H, W)?
#
# Your answer should explain what each dimension means and why convolution
# layers need the channel dimension separated from height and width.

Exercise 3 Reference Answer

Image tensors are usually shaped `(N, C, H, W)` because convolution layers process a batch of `N` images, with channels separated from spatial height and width.

## Summary

At this point, you have completed your first full CNN image-classification workflow. You connected image preprocessing, a custom image dataset, DataLoaders, CNN construction, training, test evaluation, and single-image inference.

The most important habit is shape tracing. The model receives `(N, C, H, W)` image batches, changes channels and spatial dimensions through convolution and pooling, then produces `(N, num_classes)` logits for classification.